In [ ]:
import numpy as np
from pathlib import Path
from scipy import linalg
import pandas as pd

In [ ]:
def build_convolution_matrix(alpha_hist: np.ndarray) -> np.ndarray:
    """
    Build a lower-triangular convolution matrix from an input history.

    Parameters
    ----------
    alpha_hist : ndarray
        Input history array (shape: n,).

    Returns
    -------
    ndarray
        Lower-triangular convolution matrix (shape: n, n).
    """
    n = alpha_hist.size
    U = np.zeros((n, n))
    for k in range(n):
        U += alpha_hist[k] * np.diag(np.ones(n - k), -k)
    return U

In [ ]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

chord = 0.4064  # m
area = 2 * chord**2
sample_step = 18
n_coeff = 17
otref = 500.0  # 1/s
dt = 5e-5
factor = 2.0

# -----------------------------------------------------------------------------
# Load data and kernels
# -----------------------------------------------------------------------------

data_dir = Path("Dataset/CFD")
kernel_dir = Path("Dataset/Volterra")
pred_dir = Path("Dataset/Predictions")

# CFD input data
dataset_CL0_CM0_M_AoA = np.load(data_dir / "dataset_CL0_CM0_M_AoA.npy").transpose(0, 2, 1)
CL_1_deg = np.load(data_dir / "dataset_CL_CM_1_deg.npy").transpose(0, 2, 1)[:, 0, :]

Mach = dataset_CL0_CM0_M_AoA[:, 0, 0]
AoA = dataset_CL0_CM0_M_AoA[:, 1, 0]
cl0 = dataset_CL0_CM0_M_AoA[:, 2, :]
cm0 = dataset_CL0_CM0_M_AoA[:, 3, :]

x_30 = 0.12192 # 30% chord position
x_50 = 0.2032 # 50% chord position
cm0 += (x_50 - x_30) * cl0 / chord # Adjusting cm0 based on cl0

# Volterra kernels
volterra = np.load(kernel_dir / "kernels_CL_CM_linear.npy")
volterra_NL = np.load(kernel_dir / "kernels_CL_CM_NL.npy")

# FCNN and GP kernel predictions
kernels_CL_linear_pred = np.load(pred_dir / "kernels_CL_linear_pred.npy")
kernels_CL_linear_pred_GP = np.load(pred_dir / "kernels_CL_linear_pred_GP.npy")
kernels_CM_linear_pred = np.load(pred_dir / "kernels_CM_linear_pred.npy")
kernels_CM_linear_pred_GP = np.load(pred_dir / "kernels_CM_linear_pred_GP.npy")
kernels_CL_NL_pred = np.load(pred_dir / "kernels_CL_NL_pred.npy")
kernels_CL_NL_pred_GP = np.load(pred_dir / "kernels_CL_NL_pred_GP.npy")
kernels_CM_NL_pred = np.load(pred_dir / "kernels_CM_NL_pred.npy")
kernels_CM_NL_pred_GP = np.load(pred_dir / "kernels_CM_NL_pred_GP.npy")

In [ ]:
# -----------------------------------------------------------------------------
# Compute CL and CM
# -----------------------------------------------------------------------------

CL_volterra, CM_volterra = [], []
CL_fcnn, CM_fcnn = [], []
CL_gp, CM_gp = [], []

for i in range(CL_1_deg.shape[0]):
    V = Mach[i] * np.sqrt(1.116 * 81.49 * 304.2128)
    it_all = np.arange(cl0.shape[1])
    time = it_all * dt
    aoa = (2.0 * np.pi / 180.0) * (1.0 - np.exp(-time * otref))
    alpha_hist = aoa[::sample_step]
    U = build_convolution_matrix(alpha_hist)

    # Volterra
    cl_lin_vol = U @ volterra[i, :, 0]
    cm_lin_vol = U @ volterra[i, :, 1]
    CL_volterra.append(cl_lin_vol * factor + (U @ volterra_NL[i, :, 0]))
    CM_volterra.append(cm_lin_vol * factor + (U @ volterra_NL[i, :, 1]))

    # FCNN
    cl_lin_fcnn = U @ kernels_CL_linear_pred[i, :, 0]
    cm_lin_fcnn = U @ kernels_CM_linear_pred[i, :, 0]
    cl_nl_fcnn = U @ kernels_CL_NL_pred[i, :, 0]
    cm_nl_fcnn = U @ kernels_CM_NL_pred[i, :, 0]
    CL_fcnn.append(cl_lin_fcnn * factor + cl_nl_fcnn)
    CM_fcnn.append(cm_lin_fcnn * factor + cm_nl_fcnn)

    # GP
    cl_lin_gp = U @ kernels_CL_linear_pred_GP[i, :, 0]
    cm_lin_gp = U @ kernels_CM_linear_pred_GP[i, :, 0]
    cl_nl_gp = U @ kernels_CL_NL_pred_GP[i, :, 0]
    cm_nl_gp = U @ kernels_CM_NL_pred_GP[i, :, 0]
    CL_gp.append(cl_lin_gp * factor + cl_nl_gp)
    CM_gp.append(cm_lin_gp * factor + cm_nl_gp)

CL_fcnn = np.array(CL_fcnn)
CM_fcnn = np.array(CM_fcnn)
CL_gp = np.array(CL_gp)
CM_gp = np.array(CM_gp)

In [ ]:
# -----------------------------------------------------------------------------
# Flutter analysis using FCNN kernels
# -----------------------------------------------------------------------------

q_flutter = []

for i in range(CL_fcnn.shape[0]):
    V = Mach[i] * np.sqrt(1.116 * 81.49 * 304.2128)  # Calculate velocity based on Mach number
    rho = 1.1751  # Air density (kg/m^3)
    q = 0.5 * rho * V**2  # Dynamic pressure

    theta_cl = kernels_CL_NL_pred[i, :, 0]  # Nonlinear lift kernel coefficients
    theta_cm = kernels_CM_NL_pred[i, :, 0]  # Nonlinear moment kernel coefficients

    ndof = len(theta_cl) + 4  # Total number of degrees of freedom
    ndof_aero = len(theta_cl)  # Number of aerodynamic degrees of freedom

    Iy = 3.765  # Pitch moment of inertia
    mass = 87.91  # Mass
    kz = mass * (3.33 * 2 * np.pi)**2  # Heave stiffness
    ktheta = Iy * (5.20 * 2 * np.pi)**2  # Pitch stiffness

    K_aa = np.array([[ktheta, 0.0], [0.0, kz]])  # Stiffness matrix
    M_aa = np.array([[Iy, 0.0], [0.0, mass]])  # Mass matrix
    xi = 0.0  # Damping ratio
    C_aa = 2 * xi * np.diag([np.sqrt(Iy * ktheta), np.sqrt(mass * kz)])  # Damping matrix

    Aphys = np.block([[np.zeros((2, 2)), np.eye(2)], [-K_aa, -C_aa]])  # Physical system A matrix
    Bphys = np.block([[np.eye(2), np.zeros((2, 2))], [np.zeros((2, 2)), M_aa]])  # Physical system B matrix

    DT = (sample_step * dt * chord) / (2 * V)  # Time step for discretization
    beta = 0.5  # Newmark-beta parameter

    Aphys[2, 0] += q * area * chord * theta_cm[0] * beta  # Aerodynamic moment effect on pitch
    Aphys[3, 0] += q * area * theta_cl[0] * beta  # Aerodynamic lift effect on heave
    Aphys[2, 3] += -q * area * chord * theta_cm[0] * beta / V  # Unsteady aerodynamic moment
    Aphys[3, 3] += -q * area * theta_cl[0] * beta / V  # Unsteady aerodynamic lift

    A_D = Bphys + DT * (1 - beta) * Aphys  # Discretized A matrix
    B_D = Bphys - DT * beta * Aphys  # Discretized B matrix
    AA_D = linalg.inv(B_D) @ A_D  # State transition matrix

    BZ = np.zeros((4, ndof_aero))  # Aerodynamic input matrix
    BZ[2, :] = q * area * chord * theta_cm[:ndof_aero]  # Aerodynamic moment input
    BZ[3, :] = q * area * theta_cl[:ndof_aero]  # Aerodynamic lift input
    BZ_D = linalg.inv(B_D) @ BZ  # Discretized aerodynamic input

    AA = np.zeros((ndof, ndof))  # Full state-space matrix
    AA[:4, :4] = AA_D  # Physical system block
    AA[:4, 4:] = DT * BZ_D  # Aerodynamic block
    AA[4, 0] = 1.0  # Initial condition for aerodynamic states
    AA[4:, 4:] = np.diag(np.ones(ndof_aero - 1), -1)  # Shift register for aerodynamic states

    nt = 12500  # Number of time steps
    state = np.zeros((ndof, nt))  # State history array
    state[2, 0] = 1.0  # Initial pitch disturbance
    state[3, 0] = 1.0  # Initial heave disturbance

    found_flutter = False  # Flag for flutter detection
    for t in range(1, nt):
        state[:, t] = AA @ state[:, t - 1]  # State propagation

    try:
        peaks = []
        for j in range(5, nt - 5):
            if state[0, j] == state[0, j - 5:j + 5].max():  # Find local maxima in pitch response
                peaks.append(j)
        peaks = peaks[4::3]  # Subsample peaks for fitting
        a, b = np.polyfit(np.arange(len(peaks)), np.log(state[0, peaks]), 1)  # Fit exponential to peaks
        if -a < 0:  # Check for instability (flutter)
            q_flutter.append(q)  # Save flutter dynamic pressure
            found_flutter = True
        else:
            q_flutter.append(np.nan)  # No flutter detected
    except:
        q_flutter.append(np.nan)  # Handle fitting errors

# -----------------------------------------------------------------------------
# Save flutter results
# -----------------------------------------------------------------------------

q_flutter = np.array(q_flutter)

df_flutter = pd.DataFrame({
    'Mach': Mach,
    'AoA': AoA,
    'Q_flutter_psf': np.round(q_flutter / 47.88, 3)
})

df_flutter.to_csv("q_flutter.csv", sep=",", index=False)
print("\nFlutter predictions saved to q_flutter.csv")
